# Data Preparation — Training Manifest with Named Entities

In [ ]:
!pip install -U spacy transformers torch torchvision jiwer pandas tqdm torchcodec
!python -m spacy download en_core_web_trf   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf

## Load the Kaldi Dataset

In [ ]:
import torchaudio
from IPython.display import Audio, display


dataset = []

dataset_path = "/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/train_nodev" #train split; *_de *_fr *_es
# dev split:      /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/train_dev
# test split:     /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test
# other datasets (all with train_nodev / train_dev / test):
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/en /de/ /fr/ /es/
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/en /de/ /fr/ /es/ 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en /de/ /fr/ /es/
# Voxpopuli: /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform
    
first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))

## Load the Dev Split

In [ ]:
dev_dataset = []

dataset_path = "/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/train_dev" #dev split

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dev_dataset.append({"audio" : wav_path, "text" : text})

print(len(dataset), "train samples,", len(dev_dataset), "dev samples")

## Named Entity Extraction

In [ ]:
import spacy

nlp = spacy.load("en_core_web_trf")   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf

def extract_entities(text):
    return [e.text for e in nlp(text).ents]

def build_prompt(text):
    seen = []
    for e in extract_entities(text):
        if e not in seen:
            seen.append(e)
    if not seen:
        return None
    return "Vocabulary: " + ", ".join(seen) + "."

print(extract_entities(first_entry["text"]))
print(build_prompt(first_entry["text"]))


## Write the Training and Dev Manifests

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
MANIFEST_DIR = Path("asr_finetune")   # <-- adapt
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

train_size = 1000
dev_size = 100

def write_manifest(samples, path, size):
    counter = 0
    with open(path, "w", encoding="UTF-8") as wf:
        for sample in tqdm(samples):
            prompt = build_prompt(sample["text"])
            if prompt is None:
                continue
            entry = {"audio": sample["audio"], "text": sample["text"],
                     "prompt": prompt}
            wf.write(json.dumps(entry, ensure_ascii=False) + "\n")
            counter += 1
            if counter >= size:
                break
    return counter

train_written = write_manifest(dataset, MANIFEST_TRAIN, train_size)
dev_written = write_manifest(dev_dataset, MANIFEST_DEV, dev_size)

print(f"Wrote {train_written} entries to {MANIFEST_TRAIN}")
print(f"Wrote {dev_written} entries to {MANIFEST_DEV}")


## Inspect the Manifests

In [ ]:
for path in [MANIFEST_TRAIN, MANIFEST_DEV]:
    print("=" * 70)
    print(path)
    with open(path, encoding="UTF-8") as rf:
        for i, line in enumerate(rf):
            if i >= 5:
                break
            print(json.loads(line))